<a href="https://colab.research.google.com/github/niranjan2399/ML-and-PyTorch/blob/main/text_classifier_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from collections import Counter
from torch.utils.data import random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [59]:
dataset = load_dataset('imdb')

split = dataset['train'].train_test_split(test_size=0.1, shuffle=True, seed=42)
train_data = split['train']
validation_data = split['test']
test_data = dataset['test']

print(f"Train Data: {len(train_data)}, Val Data: {len(validation_data)}, Test Data: {len(test_data)}")

Train Data: 22500, Val Data: 2500, Test Data: 25000


In [60]:
def Tokenize(text):
  return text.lower().split()

counter = Counter()
for item in train_data:
  counter.update(Tokenize(item['text']))

vocab = {"<unk>": 0}
for word, freq in counter.items():
  if (freq >= 5): vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 43179


In [61]:
def get_indices(text):
  return [vocab.get(word, 0) for word in Tokenize(text)]

def collate_batch(batch):
  labels, texts, offsets = [], [], [0]

  for item in batch:
    labels.append(item['label'])
    indices = torch.tensor(get_indices(item['text']), dtype=torch.long)
    texts.append(indices)
    offsets.append(len(indices))

  labels = torch.tensor(labels, dtype=torch.long)
  texts = torch.cat(texts)
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)

  return labels, texts, offsets

In [62]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(validation_data, batch_size=64, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, collate_fn=collate_batch)

In [63]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        x = self.fc1(embedded)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

model = TextClassifier(vocab_size=len(vocab), embed_dim=256, num_classes=2)

In [64]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
loss_fn = nn.CrossEntropyLoss()

In [65]:
best_val_loss = float('inf')
patience = 3
patience_counter = 0

for epoch in range(50):
  model.train()
  total, total_loss, correct = 0, 0, 0

  for labels, text, offsets in train_loader:
    predictions = model(text, offsets)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += loss.item()
    correct += (predictions.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)

  model.eval()
  val_correct, val_total, val_loss = 0, 0, 0

  with torch.no_grad():
    for labels, text, offsets in val_loader:
      predictions = model(text, offsets)
      loss = loss_fn(predictions, labels)

      val_loss += loss.item()
      val_correct += (predictions.argmax(dim=1) == labels).sum().item()
      val_total += labels.size(0)

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), 'best_model.pt')
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

  scheduler.step(val_loss / val_total)
  print(f"Epoch {epoch}: train_loss={total_loss/total:.4f}, train_acc={correct/total:.4f}, val_loss={val_loss/val_total:.4f}, val_acc={val_correct/val_total:.4f}")

Epoch 0: train_loss=0.0099, train_acc=0.6434, val_loss=0.0089, val_acc=0.7232
Epoch 1: train_loss=0.0082, train_acc=0.7491, val_loss=0.0076, val_acc=0.7772
Epoch 2: train_loss=0.0071, train_acc=0.7921, val_loss=0.0068, val_acc=0.8068
Epoch 3: train_loss=0.0063, train_acc=0.8203, val_loss=0.0063, val_acc=0.8244
Epoch 4: train_loss=0.0057, train_acc=0.8420, val_loss=0.0059, val_acc=0.8392
Epoch 5: train_loss=0.0052, train_acc=0.8621, val_loss=0.0057, val_acc=0.8536
Epoch 6: train_loss=0.0048, train_acc=0.8773, val_loss=0.0055, val_acc=0.8472
Epoch 7: train_loss=0.0043, train_acc=0.8902, val_loss=0.0052, val_acc=0.8648
Epoch 8: train_loss=0.0039, train_acc=0.9023, val_loss=0.0051, val_acc=0.8656
Epoch 9: train_loss=0.0036, train_acc=0.9133, val_loss=0.0054, val_acc=0.8644
Epoch 10: train_loss=0.0033, train_acc=0.9234, val_loss=0.0049, val_acc=0.8768
Epoch 11: train_loss=0.0029, train_acc=0.9343, val_loss=0.0049, val_acc=0.8744
Epoch 12: train_loss=0.0026, train_acc=0.9427, val_loss=0.0052

In [67]:
model.load_state_dict(torch.load('best_model.pt'))

model.eval()
test_total, test_loss, test_correct = 0, 0, 0

with torch.no_grad():
  for labels, text, offsets in test_loader:
    prediction = model(text, offsets)
    loss = loss_fn(prediction, labels)

    test_loss += loss.item()
    test_correct += (prediction.argmax(dim=1) == labels).sum().item()
    test_total += labels.size(0)

print(f"Test Accuracy: {test_correct/test_total:.4f}")

Test Accuracy: 0.8671
